# Model Training & Evaluation

Implements Design Doc §5.3-§5.5 and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 4: scaffold split, XGBoost-on-ECFP vs. MLP-on-ChemBERTa, evaluated with RMSE/R²/Spearman.

**Phase 4 is complete (steps 1-7):** scaffold split, training both models, formal held-out evaluation, a direct bootstrap-backed comparison, saving models/metrics to `results/`, and diagnostic plots to `results/figures/`.

**Addendum (step 8):** a variant-aware model, added as a Phase 5 prerequisite after discovering the Phase 4 models can't structurally distinguish WT from D816V for the same compound.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("../src")
from data_utils import scaffold_split

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "kit_bioactivity_clean.csv")
print("Loaded:", df.shape)
df.head()

Loaded: (5565, 10)


,molecule_chembl_id,kit_variant,canonical_smiles,p_value,censored,censored_direction,n_measurements,p_value_std,n_documents,standard_types
0,CHEMBL10,D816V,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,True,>,3.0,NaN,3.0,Kd
1,CHEMBL10,WT,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,True,>,13.0,NaN,4.0,"Kd,Ki"
2,CHEMBL101253,D816V,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,5.000000,True,>,3.0,NaN,3.0,Kd
3,CHEMBL101253,WT,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,6.677781,False,NaN,17.0,1.308074,7.0,"IC50,Kd,Ki"
4,CHEMBL101683,WT,O=C(Nc1ccc(Cl)cc1)c1ccccc1NCc1ccncc1,6.619789,False,NaN,1.0,NaN,1.0,IC50


## 1. Scaffold split (train/test)

Design Doc §5.4: use a **scaffold split**, not a random split — grouping compounds by Bemis-Murcko scaffold before splitting, since random splits let near-identical analogues leak across train/test and overestimate generalization.

`scaffold_split` (in [`src/data_utils.py`](../src/data_utils.py)) groups row indices by scaffold, then assigns whole scaffold groups — largest first — to train until an 80% target is hit, with the remainder (smaller, rarer-scaffold groups) going to test. WT/D816V rows of the same compound share identical SMILES and therefore identical scaffolds, so they always land on the same side of the split; no special-casing needed.

In [2]:
train_idx, test_idx = scaffold_split(df["canonical_smiles"].tolist(), frac_train=0.8, seed=0)

print(f"Train: {len(train_idx)} rows ({len(train_idx) / len(df):.1%})")
print(f"Test:  {len(test_idx)} rows ({len(test_idx) / len(df):.1%})")

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

Train: 4452 rows (80.0%)
Test:  1113 rows (20.0%)


### Verify no scaffold leakage between train and test

The whole point of a scaffold split is that no scaffold appears on both sides — confirm that directly rather than trusting the split logic didn't have an off-by-one.

In [3]:
from data_utils import bemis_murcko_scaffold

train_scaffolds = set(train_df["canonical_smiles"].map(bemis_murcko_scaffold))
test_scaffolds = set(test_df["canonical_smiles"].map(bemis_murcko_scaffold))

overlap = train_scaffolds & test_scaffolds
print(f"Unique scaffolds — train: {len(train_scaffolds)}, test: {len(test_scaffolds)}")
print(f"Scaffolds appearing in both: {len(overlap)}")
assert not overlap, "Scaffold leakage between train and test!"

# Sanity-check the WT/D816V co-location invariant: every compound's rows should
# be entirely in train or entirely in test, never split across both.
split_side = pd.Series("train", index=df.index)
split_side.iloc[test_idx] = "test"
sides_per_compound = df.groupby("molecule_chembl_id").apply(
    lambda g: split_side.loc[g.index].nunique(), include_groups=False
)
straddling = sides_per_compound[sides_per_compound > 1]
print(f"Compounds with rows split across train AND test: {len(straddling)}")
assert straddling.empty, "A compound's WT/D816V rows ended up on different sides of the split!"


Unique scaffolds — train: 814, test: 1113
Scaffolds appearing in both: 0
Compounds with rows split across train AND test: 0


## 2. Baseline model: XGBoost on ECFP fingerprints

Design Doc §5.3: gradient-boosted trees on fingerprint features are a realistic, strong baseline given the likely small-to-medium dataset size, and shouldn't be skipped in favor of jumping straight to the ChemBERTa-based model.

`train_xgboost_ecfp` (in [`src/models.py`](../src/models.py)) trains on the cached ECFP fingerprints from `03_featurization.ipynb`, sliced to `train_idx`/`test_idx` from the scaffold split above. **This step only trains the model and sanity-checks that it actually fit** — the formal held-out RMSE/R²/Spearman evaluation (Phase 4 step 4) comes once the ChemBERTa/MLP model exists too, so both can be compared side by side.

In [4]:
ecfp = np.load(PROCESSED_DIR / "ecfp_fingerprints.npy")
assert ecfp.shape[0] == len(df), "ECFP cache is not row-aligned with the cleaned dataset"

y = df["p_value"].to_numpy()

X_train, X_test = ecfp[train_idx], ecfp[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)

X_train: (4452, 2048)  y_train: (4452,)
X_test:  (1113, 2048)  y_test:  (1113,)


In [5]:
from models import train_xgboost_ecfp

xgb_ecfp = train_xgboost_ecfp(X_train, y_train)
print(f"Trained XGBRegressor on {X_train.shape[0]} compounds, {X_train.shape[1]}-bit ECFP features.")

Trained XGBRegressor on 4452 compounds, 2048-bit ECFP features.


### Sanity-check: did it actually fit?

Not the formal evaluation (that's step 4) — just confirming the model learned a real, non-degenerate relationship rather than silently failing (e.g. from a features/labels misalignment) before moving on. Tree ensembles should fit training data closely; a near-zero training R² would mean something upstream is broken. A quick look at held-out predictions checks they aren't constant and are at least positively correlated with truth.

In [6]:
from sklearn.metrics import r2_score

train_r2 = r2_score(y_train, xgb_ecfp.predict(X_train))
print(f"Training-set R² (fit sanity check, NOT held-out performance): {train_r2:.3f}")
assert train_r2 > 0.5, "Model barely fit the training data -- check feature/label alignment"

test_preds = xgb_ecfp.predict(X_test)
test_corr = np.corrcoef(test_preds, y_test)[0, 1]
print(f"Held-out prediction std: {test_preds.std():.3f} (non-degenerate: not a constant prediction)")
print(f"Held-out Pearson correlation (predicted vs. actual p_value): {test_corr:.3f}")
assert test_preds.std() > 1e-3, "Model is predicting a near-constant value on held-out data"
assert test_corr > 0, "Held-out predictions are not even positively correlated with truth"

Training-set R² (fit sanity check, NOT held-out performance): 0.809
Held-out prediction std: 0.841 (non-degenerate: not a constant prediction)
Held-out Pearson correlation (predicted vs. actual p_value): 0.725


## 3. Comparison model: small MLP head on frozen ChemBERTa embeddings

Design Doc §5.3: a small MLP on top of the frozen ChemBERTa embeddings from `03_featurization.ipynb`, compared directly against the XGBoost-on-ECFP baseline above rather than assumed to win — echoing the dissertation's own finding that a simpler method can outperform a more sophisticated one on a small dataset.

`train_mlp_chemberta` (in [`src/models.py`](../src/models.py)) standardizes the embeddings first (MLP training is scale-sensitive, unlike the tree-based baseline) and uses early stopping so training doesn't run longer than needed. **Same scope as step 2: train + sanity-check only** — the formal head-to-head comparison is step 4.

In [7]:
chemberta = np.load(PROCESSED_DIR / "chemberta_embeddings.npy")
assert chemberta.shape[0] == len(df), "ChemBERTa cache is not row-aligned with the cleaned dataset"

X_train_cb, X_test_cb = chemberta[train_idx], chemberta[test_idx]
# y_train / y_test (the p_value labels) are already defined from step 2 above --
# same rows, same split, just a different feature matrix.

print("X_train_cb:", X_train_cb.shape, " y_train:", y_train.shape)
print("X_test_cb: ", X_test_cb.shape, " y_test: ", y_test.shape)

X_train_cb: (4452, 768)  y_train: (4452,)
X_test_cb:  (1113, 768)  y_test:  (1113,)


In [8]:
from models import train_mlp_chemberta

mlp_chemberta = train_mlp_chemberta(X_train_cb, y_train)
n_iter = mlp_chemberta.named_steps["mlpregressor"].n_iter_
print(f"Trained MLP on {X_train_cb.shape[0]} compounds, 768-dim ChemBERTa embeddings ({n_iter} iterations, early-stopped).")

Trained MLP on 4452 compounds, 768-dim ChemBERTa embeddings (42 iterations, early-stopped).


### Sanity-check: did it actually fit?

Same reasoning as the XGBoost sanity check above — confirm the model learned a real relationship before moving on, not the formal evaluation.

In [9]:
train_r2_mlp = r2_score(y_train, mlp_chemberta.predict(X_train_cb))
print(f"Training-set R² (fit sanity check, NOT held-out performance): {train_r2_mlp:.3f}")
assert train_r2_mlp > 0.5, "Model barely fit the training data -- check feature/label alignment"

test_preds_mlp = mlp_chemberta.predict(X_test_cb)
test_corr_mlp = np.corrcoef(test_preds_mlp, y_test)[0, 1]
print(f"Held-out prediction std: {test_preds_mlp.std():.3f} (non-degenerate: not a constant prediction)")
print(f"Held-out Pearson correlation (predicted vs. actual p_value): {test_corr_mlp:.3f}")
assert test_preds_mlp.std() > 1e-3, "Model is predicting a near-constant value on held-out data"
assert test_corr_mlp > 0, "Held-out predictions are not even positively correlated with truth"

Training-set R² (fit sanity check, NOT held-out performance): 0.748
Held-out prediction std: 1.174 (non-degenerate: not a constant prediction)
Held-out Pearson correlation (predicted vs. actual p_value): 0.562


## 4. Evaluate both models on the held-out set

Design Doc §5.5: RMSE and R² on held-out `p_value`, plus Spearman rank correlation as a secondary metric — for potency prediction, getting the *relative ranking* of compounds right often matters more in practice than the exact value.

`evaluate_regression` (in [`src/evaluation.py`](../src/evaluation.py)) computes all three from `(y_true, y_pred)`. Applied here to both models' held-out predictions from steps 2-3, on the identical 1,113-row scaffold-split test set. (Directly comparing the two — which wins and why — is step 5; this step is just computing the numbers.)

In [10]:
from evaluation import evaluate_regression

ecfp_metrics = evaluate_regression(y_test, xgb_ecfp.predict(X_test))
chemberta_metrics = evaluate_regression(y_test, mlp_chemberta.predict(X_test_cb))

metrics_df = pd.DataFrame(
    {"XGBoost + ECFP": ecfp_metrics, "MLP + ChemBERTa": chemberta_metrics}
).T
metrics_df.columns = ["RMSE", "R²", "Spearman ρ"]
metrics_df.round(3)

,RMSE,R²,Spearman ρ
XGBoost + ECFP,0.849,0.520,0.702
MLP + ChemBERTa,1.125,0.156,0.551


### Sanity-check against the naive baseline

Both models should clearly beat "always predict the training-set mean" — otherwise they aren't capturing any real structure-activity signal at all.

In [11]:
naive_baseline_pred = np.full_like(y_test, y_train.mean())
naive_rmse = np.sqrt(np.mean((y_test - naive_baseline_pred) ** 2))
print(f"Naive mean-baseline held-out RMSE: {naive_rmse:.3f} (R²=0 by construction)")
# (Spearman is undefined for a constant prediction, so skip it here rather
# than calling evaluate_regression and triggering a ConstantInputWarning for
# a value we don't need.)

for name, metrics in [("XGBoost + ECFP", ecfp_metrics), ("MLP + ChemBERTa", chemberta_metrics)]:
    assert metrics["rmse"] < naive_rmse, f"{name} failed to beat the naive mean baseline on RMSE"
    assert metrics["r2"] > 0, f"{name} has non-positive R² -- no better than predicting the mean"
print("Both models beat the naive mean-baseline on RMSE and R².")

Naive mean-baseline held-out RMSE: 1.233 (R²=0 by construction)
Both models beat the naive mean-baseline on RMSE and R².


## 5. Compare the two models directly

Design Doc §5.3: compare baseline vs. embedding model directly — don't assume the more sophisticated model wins, echoing the dissertation's own GAN-vs-augmentation finding that a simpler method can beat a fancier one on a small dataset. Step 4's single point estimate already shows XGBoost + ECFP ahead on every metric, but a single fixed 1,113-row test set could make a real gap look bigger (or smaller) than it robustly is. `bootstrap_compare` (in [`src/evaluation.py`](../src/evaluation.py)) resamples the held-out set with replacement 1,000 times and reports how often each model comes out ahead — turning "0.849 vs. 1.125" into a claim about robustness, not just one number.

In [12]:
from evaluation import bootstrap_compare

ecfp_preds = xgb_ecfp.predict(X_test)
chemberta_preds = mlp_chemberta.predict(X_test_cb)

win_rates = bootstrap_compare(y_test, ecfp_preds, chemberta_preds, n_boot=1000, seed=0)
print("Fraction of 1,000 bootstrap resamples where XGBoost + ECFP beat MLP + ChemBERTa:")
for metric, win_rate in win_rates.items():
    print(f"  {metric}: {win_rate:.3f}")

Fraction of 1,000 bootstrap resamples where XGBoost + ECFP beat MLP + ChemBERTa:
  rmse: 1.000
  r2: 1.000
  spearman: 1.000


### Discussion: why did the simpler baseline win here?

XGBoost + ECFP beats MLP + ChemBERTa on RMSE, R², and Spearman ρ in essentially every bootstrap resample — this isn't a fragile, one-test-set artifact. A few concrete, non-mutually-exclusive reasons this particular comparison likely came out this way:

- **ChemBERTa is frozen, not fine-tuned.** It was pretrained on ~100M generic ZINC15 molecules for general chemical structure, not on kinase-inhibitor binding specifically. Without fine-tuning, its embeddings carry broad chemical-similarity signal but nothing tailored to *this* binding site — the MLP head has to extract task-specific structure-activity signal from a representation that wasn't built for the task.
- **Sample size favors the classical method.** ~4,452 training compounds is a small-to-medium dataset for training a neural network from a 768-dim representation; gradient-boosted trees on a sparse, high-dimensional binary fingerprint (2,048 bits) are a well-established strong, sample-efficient baseline in exactly this regime — which is precisely why Design Doc §5.3 called for including it rather than skipping straight to the embedding model.
- **ECFP substructure bits are close to directly interpretable pharmacophore signal** (specific substructure fragments known to matter for kinase binding), whereas a mean-pooled sentence-level embedding is a more diffuse, indirect encoding of the same molecule.

**Important scope caveat:** this result is specific to *this* setup — no hyperparameter tuning on either model, and ChemBERTa used strictly frozen rather than fine-tuned. It's evidence for "don't assume sophistication wins by default here," not a general claim that fingerprints beat pretrained embeddings on every problem or that fine-tuning ChemBERTa wouldn't close the gap.

## 6. Save trained models and evaluation metrics

Persist both fitted models (`results/models/`, via `joblib`) and the comparison table (`results/model_comparison.csv`) so later notebooks (e.g. Phase 5's selectivity analysis) can load the trained models directly instead of retraining, and so the metrics are available as a durable artifact rather than only living in this notebook's output cells.

Model files are reproducible by re-running this notebook (fixed `random_state`/seeds throughout) and are gitignored accordingly, matching the existing convention for `data/raw/` and `data/processed/`. `results/model_comparison.csv` is the actual small deliverable and is tracked in git.

In [13]:
import joblib

RESULTS_DIR = Path("../results")
MODELS_DIR = RESULTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(xgb_ecfp, MODELS_DIR / "xgb_ecfp.joblib")
joblib.dump(mlp_chemberta, MODELS_DIR / "mlp_chemberta.joblib")

print(f"Saved {MODELS_DIR / 'xgb_ecfp.joblib'}")
print(f"Saved {MODELS_DIR / 'mlp_chemberta.joblib'}")

Saved ../results/models/xgb_ecfp.joblib
Saved ../results/models/mlp_chemberta.joblib


### Verify the saved models actually reload correctly

Confirm the round-tripped models predict identically to the in-memory ones — not just that `joblib.dump` ran without an error.

In [14]:
reloaded_xgb = joblib.load(MODELS_DIR / "xgb_ecfp.joblib")
reloaded_mlp = joblib.load(MODELS_DIR / "mlp_chemberta.joblib")

assert np.array_equal(reloaded_xgb.predict(X_test), ecfp_preds), "Reloaded XGBoost model predicts differently!"
assert np.array_equal(reloaded_mlp.predict(X_test_cb), chemberta_preds), "Reloaded MLP model predicts differently!"
print("Both reloaded models reproduce the original in-memory predictions exactly.")

Both reloaded models reproduce the original in-memory predictions exactly.


### Write the comparison table to `results/`

Combines step 4's RMSE/R²/Spearman with step 5's bootstrap win-rates into one durable, trackable artifact.

In [15]:
comparison_table = metrics_df.copy()
comparison_table["bootstrap_win_rate_vs_other_model"] = [
    win_rates["rmse"],  # XGBoost + ECFP's win rate (computed as model A in step 5)
    1 - win_rates["rmse"],  # MLP + ChemBERTa's complementary win rate
]

comparison_path = RESULTS_DIR / "model_comparison.csv"
comparison_table.to_csv(comparison_path)
print(f"Saved {comparison_path}")
comparison_table

Saved ../results/model_comparison.csv


,RMSE,R²,Spearman ρ,bootstrap_win_rate_vs_other_model
XGBoost + ECFP,0.849183,0.519652,0.701822,1.0
MLP + ChemBERTa,1.125421,0.156309,0.551134,0.0


## 7. Diagnostic plots

Design Doc §5.5 / IMPLEMENTATION_PLAN Phase 4 step 7: predicted-vs-actual and residual plots for both models, saved to `results/figures/`.

`plot_diagnostics` (in [`src/evaluation.py`](../src/evaluation.py)) draws each model as its own column — predicted vs. actual (with a dashed y=x parity line) on top, residuals vs. predicted (with a dashed zero line) below — using a fixed, colorblind-validated color per model (Okabe-Ito blue/vermillion, checked with the dataviz skill's `validate_palette.js`).

In [16]:
from evaluation import plot_diagnostics

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig = plot_diagnostics(
    y_test,
    {"XGBoost + ECFP": ecfp_preds, "MLP + ChemBERTa": chemberta_preds},
    save_path=FIGURES_DIR / "model_diagnostics.png",
)
print(f"Saved {FIGURES_DIR / 'model_diagnostics.png'}")

Saved ../results/figures/model_diagnostics.png


### Reading the plots

- **Predicted vs. actual**: XGBoost's predictions are visibly more compressed (roughly 4–8.3) than the MLP's (roughly 2.7–10.4) — tree ensembles can't extrapolate past the range of training targets, while the MLP's linear output layer can. This is concrete visual evidence for the extrapolation-related reasoning in step 5's discussion, not just an abstract claim.
- **The diagonal/banded streaks visible in both models' plots are a real data feature, not a plotting artifact.** Many compounds share the *exact same* `p_value` — e.g. 569 rows at 6.301030, 453 at 5.899998, 259 at exactly 5.0 — because Phase 2 capped censored measurements (">10000 nM" etc.) to common threshold values rather than dropping them. A cluster of test compounds sharing the same true value, combined with each model producing a locally clustered set of predictions for structurally-related compounds, produces the visible banding. Verified directly: `(df["p_value"] == 5.0).sum()` → 259 rows, `df["censored"].sum()` → 1,475/5,565 rows.
- Both residual plots are reasonably centered on zero with no obvious strong trend, though XGBoost's residual spread is visibly tighter — consistent with its better RMSE/R² from step 4.

## 8. Addendum: variant-aware model (Phase 5 prerequisite)

Phase 4's models above predict potency from structure alone (ECFP / ChemBERTa). That's a problem for Phase 5 (selectivity analysis): a compound with both a WT and a D816V bioactivity record has *identical* structural features for both rows, so the Phase 4 model literally cannot distinguish them -- it necessarily predicts the same value regardless of which KIT variant is being asked about. Confirmed directly: imatinib's WT and D816V rows (identical fingerprint) would get one shared prediction from the Phase 4 model, even though their actual measured potencies differ (WT p_value=6.97, D816V p_value=6.01).

Fix: `add_variant_indicator` (in [`src/featurization.py`](../src/featurization.py)) appends a binary WT/D816V flag to the feature matrix. Retraining XGBoost-on-ECFP (the step 5 winner) with this flag added gives the model the information it needs to learn a variant-dependent shift. This is scoped narrowly to the model Phase 5 will actually use -- not a full redo of the Phase 4 ECFP-vs-ChemBERTa comparison, which remains valid on its own terms (it was answering a different question: which *representation* predicts potency better, not whether variant is encoded).

In [17]:
from featurization import add_variant_indicator

ecfp_variant = add_variant_indicator(ecfp, df["kit_variant"].to_numpy())
print("Augmented feature shape:", ecfp_variant.shape, "(was", ecfp.shape, ")")

X_train_v, X_test_v = ecfp_variant[train_idx], ecfp_variant[test_idx]
xgb_ecfp_variant = train_xgboost_ecfp(X_train_v, y_train)
print(f"Trained variant-aware XGBRegressor on {X_train_v.shape[0]} compounds, {X_train_v.shape[1]} features.")

Augmented feature shape: (5565, 2049) (was (5565, 2048) )


Trained variant-aware XGBRegressor on 4452 compounds, 2049 features.


### Verify the fix actually works

Two checks: (1) held-out metrics shouldn't get worse -- if resolving the structural ambiguity hurts generalization, that's a red flag; (2) the model must now produce genuinely different predictions for the same compound's WT vs. D816V rows, which was structurally impossible before.

In [18]:
variant_metrics = evaluate_regression(y_test, xgb_ecfp_variant.predict(X_test_v))
print("Variant-aware held-out metrics:", {k: round(v, 3) for k, v in variant_metrics.items()})
print("Original (structure-only) held-out metrics:", {k: round(v, 3) for k, v in ecfp_metrics.items()})

for metric in ("rmse",):
    assert variant_metrics[metric] <= ecfp_metrics[metric] + 0.05, (
        f"Adding the variant flag made {metric} meaningfully worse -- investigate before using this model"
    )

# Same compound, both variants: predictions must now differ (impossible for the Phase 4 model).
imatinib_rows = df.index[df["molecule_chembl_id"] == "CHEMBL941"]
imatinib_preds = pd.Series(
    xgb_ecfp_variant.predict(ecfp_variant[imatinib_rows]),
    index=df.loc[imatinib_rows, "kit_variant"],
)
print("\nImatinib predictions by variant (now distinguishable):")
print(imatinib_preds)
assert imatinib_preds.nunique() > 1, "Model still predicts identically across variants -- fix didn't work"

Variant-aware held-out metrics: {'rmse': 0.843, 'r2': 0.526, 'spearman': 0.707}
Original (structure-only) held-out metrics: {'rmse': 0.849, 'r2': 0.52, 'spearman': 0.702}

Imatinib predictions by variant (now distinguishable):
kit_variant
D816V    6.801572
WT       7.013057
dtype: float32


### Save the variant-aware model

Saved separately from the Phase 4 models (`xgb_ecfp_variant_aware.joblib`, not overwriting `xgb_ecfp.joblib`) since it solves a different problem and takes a different input shape (2,049 features vs. 2,048).

**Caveat carried into Phase 5, stated explicitly rather than glossed over:** both anchor compounds (imatinib, dasatinib) are in the *train* set under this scaffold split -- confirmed directly, not assumed. So checking whether this model reproduces their known WT-vs-D816V direction is evaluating in-sample fit for those two specific compounds, not genuine held-out generalization. That's consistent with how Design Doc §5.6/§9 already frames this whole exercise (a validation check on a handful of named anchors, not a rigorous benchmark) -- but it's worth being explicit that "reproduces the known direction" here is a weaker claim than it would be for a compound the model never trained on. No paired WT/D816V compound currently lands in the test set, so this limitation can't be fully sidestepped with the current split.

In [19]:
joblib.dump(xgb_ecfp_variant, MODELS_DIR / "xgb_ecfp_variant_aware.joblib")
print(f"Saved {MODELS_DIR / 'xgb_ecfp_variant_aware.joblib'}")

reloaded_variant = joblib.load(MODELS_DIR / "xgb_ecfp_variant_aware.joblib")
assert np.array_equal(reloaded_variant.predict(X_test_v), xgb_ecfp_variant.predict(X_test_v))
print("Reloaded variant-aware model reproduces predictions exactly.")

Saved ../results/models/xgb_ecfp_variant_aware.joblib
Reloaded variant-aware model reproduces predictions exactly.


## 9. Addendum: hyperparameter tuning for the best legitimate accuracy

The untuned XGBoost-on-ECFP baseline above used reasonable defaults, never tuned. This section tunes it properly via cross-validation and reports the real, single, final number on the held-out test set.

**Ceiling, stated up front:** Phase 2 found that ChEMBL's own repeated measurements of the *same* compound disagree by 0.45–0.8 log units across assay types (cross-type divergence, §2 of that notebook). That's a noise floor baked into the labels themselves — no amount of tuning can predict past the noise in what's being predicted. So the realistic goal here is "as good as this data legitimately supports," not "as close to perfect as possible." An RMSE approaching that ~0.5–0.8 range would be near the ceiling of what's achievable; anything claiming much better than that on this data would be a red flag (leakage or a bug), not a win.

**Method, to avoid a subtler version of the exact leakage problem the scaffold split exists to prevent:** tuning via cross-validation on the training set needs its own CV folds, and a plain random `KFold` would let near-identical analogues split across a fold's train/validation halves — the same optimism-inflating leakage Design Doc §5.4 flags for the outer train/test split. `tune_xgboost_ecfp` (added to [`src/models.py`](../src/models.py)) uses `GroupKFold` grouped by Bemis-Murcko scaffold instead, so no scaffold crosses a fold boundary. **The test set is never touched by the search** — only used once, at the very end, to report the final number.

In [20]:
from data_utils import bemis_murcko_scaffold
from models import tune_xgboost_ecfp

train_smiles = df["canonical_smiles"].to_numpy()[train_idx]
train_scaffold_groups = np.array([bemis_murcko_scaffold(s) for s in train_smiles])
print(f"{len(set(train_scaffold_groups))} unique scaffolds across {len(train_scaffold_groups)} training rows "
      "(grouped CV keeps every scaffold's rows in one fold, never split across train/validation within a fold).")

xgb_ecfp_tuned, tuned_params = tune_xgboost_ecfp(
    X_train, y_train, train_scaffold_groups, n_iter=80, n_splits=5, seed=0, n_jobs=8
)
print("\nBest hyperparameters found (80 candidates x 5 scaffold-grouped folds = 400 fits, train set only):")
for key, value in tuned_params.items():
    print(f"  {key}: {value}")

814 unique scaffolds across 4452 training rows (grouped CV keeps every scaffold's rows in one fold, never split across train/validation within a fold).



Best hyperparameters found (80 candidates x 5 scaffold-grouped folds = 400 fits, train set only):
  colsample_bytree: 0.3278960537143403
  gamma: 0.25882467484450267
  learning_rate: 0.11999525363238997
  max_depth: 8
  min_child_weight: 8
  n_estimators: 838
  reg_alpha: 0.9172079235371717
  reg_lambda: 7.379592547809661
  subsample: 0.699512660851551


### Evaluate once on the held-out test set

The one and only time the test set is used in this whole tuning process.

In [21]:
tuned_metrics = evaluate_regression(y_test, xgb_ecfp_tuned.predict(X_test))
tuned_train_metrics = evaluate_regression(y_train, xgb_ecfp_tuned.predict(X_train))

comparison = pd.DataFrame(
    {
        "XGBoost + ECFP (untuned, Phase 4)": ecfp_metrics,
        "XGBoost + ECFP (tuned)": tuned_metrics,
    }
).T
comparison.columns = ["RMSE", "R²", "Spearman ρ"]
print("Held-out test set:")
display(comparison.round(3))

print(f"\nTuned model train-set RMSE: {tuned_train_metrics['rmse']:.3f} vs. test-set RMSE: {tuned_metrics['rmse']:.3f} "
      f"-- the gap is the honest generalization cost of a scaffold split.")

improvement_pct = (ecfp_metrics["rmse"] - tuned_metrics["rmse"]) / ecfp_metrics["rmse"] * 100
print(f"\nRMSE improved by {improvement_pct:.1f}% relative to the untuned baseline -- "
      "real, from proper CV tuning alone, no test-set contact during the search, and consistent with "
      "the ~0.5-0.8 log-unit noise floor discussed above (a modest gain given how much of the remaining "
      "error is likely irreducible measurement noise, not model error).")

assert tuned_metrics["rmse"] < ecfp_metrics["rmse"], "Tuning made held-out RMSE worse -- investigate before using this model"

Held-out test set:


,RMSE,R²,Spearman ρ
"XGBoost + ECFP (untuned, Phase 4)",0.849,0.520,0.702
XGBoost + ECFP (tuned),0.830,0.541,0.714



Tuned model train-set RMSE: 0.483 vs. test-set RMSE: 0.830 -- the gap is the honest generalization cost of a scaffold split.

RMSE improved by 2.3% relative to the untuned baseline -- real, from proper CV tuning alone, no test-set contact during the search, and consistent with the ~0.5-0.8 log-unit noise floor discussed above (a modest gain given how much of the remaining error is likely irreducible measurement noise, not model error).


### Retrain the variant-aware model with the tuned hyperparameters

The variant-aware model (Phase 5's dependency) uses the same 2,048 ECFP bits plus one extra flag column. Rather than run a second full search on nearly-identical feature space, the tuned hyperparameters found above are transferred directly and used to retrain on the 2,049-feature variant-augmented input -- a reasonable, explicitly-stated simplification, not a second blind guess.

In [22]:
from models import predict_both_variants

xgb_ecfp_variant_tuned = train_xgboost_ecfp(X_train_v, y_train, **tuned_params)

variant_tuned_metrics = evaluate_regression(y_test, xgb_ecfp_variant_tuned.predict(X_test_v))
print("Variant-aware model, tuned hyperparameters, held-out test set:")
print({k: round(v, 3) for k, v in variant_tuned_metrics.items()})
print("Variant-aware model, untuned (from the addendum earlier in this notebook):")
print({k: round(v, 3) for k, v in variant_metrics.items()})

assert variant_tuned_metrics["rmse"] < variant_metrics["rmse"], (
    "Tuned variant-aware model is worse than the untuned one -- investigate before replacing it"
)

# Re-verify the anchor checks still hold with the newly tuned variant-aware model.
imatinib_row_idx = df.index[df["molecule_chembl_id"] == "CHEMBL941"][0]
dasatinib_row_idx = df.index[df["molecule_chembl_id"] == "CHEMBL1421"][0]
im_pred_wt, im_pred_d816v = predict_both_variants(xgb_ecfp_variant_tuned, ecfp[imatinib_row_idx])
das_pred_wt, das_pred_d816v = predict_both_variants(xgb_ecfp_variant_tuned, ecfp[dasatinib_row_idx])

im_shift = (im_pred_wt - im_pred_d816v)[0]
das_shift = (das_pred_wt - das_pred_d816v)[0]
print(f"\nRe-checked with the tuned model -- imatinib predicted shift: {im_shift:.3f} log units "
      f"(fold={10**im_shift:.2f}x), dasatinib: {das_shift:.3f} log units (fold={10**das_shift:.2f}x)")

assert im_shift > 0, "Tuned model no longer predicts imatinib WT-favoring -- Phase 5's anchor check would break"
assert im_shift > das_shift, "Tuned model no longer predicts imatinib's shift as bigger than dasatinib's"
print("Both Phase 5 anchor-direction checks still pass with the tuned model.")

Variant-aware model, tuned hyperparameters, held-out test set:
{'rmse': 0.824, 'r2': 0.548, 'spearman': 0.719}
Variant-aware model, untuned (from the addendum earlier in this notebook):
{'rmse': 0.843, 'r2': 0.526, 'spearman': 0.707}

Re-checked with the tuned model -- imatinib predicted shift: 0.595 log units (fold=3.93x), dasatinib: 0.136 log units (fold=1.37x)
Both Phase 5 anchor-direction checks still pass with the tuned model.


### Replace the production models and comparison table

The tuned models are strictly better on held-out data (verified above, not assumed) and pass the same anchor checks, so they replace the untuned ones as the models this project actually uses going forward. Saved under the *same* filenames Phase 5/6 already reference, so those notebooks pick up the improved model automatically next time they're run -- re-running them is the next step after this notebook.

In [23]:
joblib.dump(xgb_ecfp_tuned, MODELS_DIR / "xgb_ecfp.joblib")
joblib.dump(xgb_ecfp_variant_tuned, MODELS_DIR / "xgb_ecfp_variant_aware.joblib")
print(f"Overwrote {MODELS_DIR / 'xgb_ecfp.joblib'} and {MODELS_DIR / 'xgb_ecfp_variant_aware.joblib'} with the tuned models.")

reloaded_tuned = joblib.load(MODELS_DIR / "xgb_ecfp.joblib")
assert np.array_equal(reloaded_tuned.predict(X_test), xgb_ecfp_tuned.predict(X_test))
print("Reloaded tuned model reproduces predictions exactly.")

final_comparison = pd.DataFrame(
    {
        "XGBoost + ECFP (tuned)": tuned_metrics,
        "MLP + ChemBERTa": chemberta_metrics,
    }
).T
final_comparison.columns = ["RMSE", "R²", "Spearman ρ"]
final_win_rates = bootstrap_compare(y_test, xgb_ecfp_tuned.predict(X_test), chemberta_preds, n_boot=1000, seed=0)
final_comparison["bootstrap_win_rate_vs_other_model"] = [final_win_rates["rmse"], 1 - final_win_rates["rmse"]]

comparison_path = RESULTS_DIR / "model_comparison.csv"
final_comparison.to_csv(comparison_path)
print(f"\nUpdated {comparison_path} with the tuned model:")
final_comparison.round(3)

Overwrote ../results/models/xgb_ecfp.joblib and ../results/models/xgb_ecfp_variant_aware.joblib with the tuned models.
Reloaded tuned model reproduces predictions exactly.



Updated ../results/model_comparison.csv with the tuned model:


,RMSE,R²,Spearman ρ,bootstrap_win_rate_vs_other_model
XGBoost + ECFP (tuned),0.830,0.541,0.714,1.0
MLP + ChemBERTa,1.125,0.156,0.551,0.0


In [24]:
# Regenerate the diagnostic plot (step 7) with the tuned model's predictions,
# so the saved figure matches the model actually saved to results/models/.
fig = plot_diagnostics(
    y_test,
    {"XGBoost + ECFP (tuned)": xgb_ecfp_tuned.predict(X_test), "MLP + ChemBERTa": chemberta_preds},
    save_path=FIGURES_DIR / "model_diagnostics.png",
)
print(f"Re-saved {FIGURES_DIR / 'model_diagnostics.png'} using the tuned model.")

Re-saved ../results/figures/model_diagnostics.png using the tuned model.


## 10. Addendum: ensembling XGBoost + ECFP with MLP + ChemBERTa

Steps 5/9 established that XGBoost + ECFP (tuned) is clearly the stronger single model. But the two models use *different* representations (structural fingerprints vs. a pretrained chemical language model) and could plausibly make partly uncorrelated errors, in which case a weighted average of their predictions might beat either model alone -- a standard, cheap thing to check before assuming a single model is the ceiling.

**Method, to avoid yet another version of the same leakage problem:** the ensemble weight alpha (in `alpha * XGBoost_pred + (1 - alpha) * MLP_pred`) is itself a parameter being chosen from data, so picking it by minimizing RMSE *on the test set* would be a subtle form of tuning against the final evaluation set -- the same mistake a held-out split exists to prevent. Instead, alpha is chosen via 5-fold **scaffold-grouped** out-of-fold (OOF) cross-validation predictions on the *training* set only (`sklearn.model_selection.cross_val_predict` with the same `GroupKFold` grouping used for hyperparameter tuning in step 9), and the test set is touched exactly once at the end, exactly as in step 9. `find_best_ensemble_weight` (added to [`src/models.py`](../src/models.py)) does the alpha grid search given any two OOF prediction arrays and the true values.

In [25]:
from sklearn.model_selection import GroupKFold, cross_val_predict

from models import _build_mlp_chemberta, _build_xgb_ecfp, find_best_ensemble_weight

cv = GroupKFold(n_splits=5)

xgb_oof = cross_val_predict(
    _build_xgb_ecfp(**tuned_params), X_train, y_train, cv=cv, groups=train_scaffold_groups, n_jobs=-1
)
mlp_oof = cross_val_predict(
    _build_mlp_chemberta(), X_train_cb, y_train, cv=cv, groups=train_scaffold_groups, n_jobs=-1
)

xgb_oof_rmse = np.sqrt(np.mean((y_train - xgb_oof) ** 2))
mlp_oof_rmse = np.sqrt(np.mean((y_train - mlp_oof) ** 2))
print(f"Scaffold-grouped 5-fold OOF RMSE on the training set (not the test set):")
print(f"  XGBoost + ECFP (tuned hyperparameters): {xgb_oof_rmse:.3f}")
print(f"  MLP + ChemBERTa:                        {mlp_oof_rmse:.3f}")

best_alpha, best_oof_rmse = find_best_ensemble_weight(y_train, xgb_oof, mlp_oof, step=0.01)
print(f"\nBest ensemble weight found via OOF grid search: alpha={best_alpha:.2f} "
      f"(alpha * XGBoost + (1 - alpha) * MLP), OOF RMSE={best_oof_rmse:.3f}")
print(f"OOF RMSE improvement over the single best OOF model: "
      f"{(min(xgb_oof_rmse, mlp_oof_rmse) - best_oof_rmse) / min(xgb_oof_rmse, mlp_oof_rmse) * 100:.1f}%")

Scaffold-grouped 5-fold OOF RMSE on the training set (not the test set):
  XGBoost + ECFP (tuned hyperparameters): 0.806
  MLP + ChemBERTa:                        1.063

Best ensemble weight found via OOF grid search: alpha=0.87 (alpha * XGBoost + (1 - alpha) * MLP), OOF RMSE=0.800
OOF RMSE improvement over the single best OOF model: 0.8%


### Evaluate the ensemble once on the held-out test set

Applies the OOF-selected `best_alpha` to the two *already-trained* final models' test predictions (`xgb_ecfp_tuned`, `mlp_chemberta` -- the same fitted models used everywhere else in this notebook, not refit). The test set is touched exactly once here, for the ensemble, exactly as step 9 touches it once for the tuned single model.

In [26]:
tuned_ecfp_preds = xgb_ecfp_tuned.predict(X_test)
ensemble_preds = best_alpha * tuned_ecfp_preds + (1 - best_alpha) * chemberta_preds
ensemble_metrics = evaluate_regression(y_test, ensemble_preds)

ensemble_comparison = pd.DataFrame(
    {
        "XGBoost + ECFP (tuned)": tuned_metrics,
        "MLP + ChemBERTa": chemberta_metrics,
        f"Ensemble (alpha={best_alpha:.2f})": ensemble_metrics,
    }
).T
ensemble_comparison.columns = ["RMSE", "R²", "Spearman ρ"]
print("Held-out test set:")
display(ensemble_comparison.round(3))

ens_win_rates = bootstrap_compare(y_test, ensemble_preds, tuned_ecfp_preds, n_boot=1000, seed=0)
print("\nFraction of 1,000 bootstrap resamples where the ensemble beat the single best model (tuned XGBoost + ECFP):")
for metric, win_rate in ens_win_rates.items():
    print(f"  {metric}: {win_rate:.3f}")

ensemble_helps = ensemble_metrics["rmse"] < tuned_metrics["rmse"] and ens_win_rates["rmse"] > 0.5
print(f"\nVerdict: ensembling {'helps' if ensemble_helps else 'does not help'} on this held-out set -- "
      f"reporting the real number either way, not adopting the ensemble just because it was tried.")

Held-out test set:


,RMSE,R²,Spearman ρ
XGBoost + ECFP (tuned),0.830,0.541,0.714
MLP + ChemBERTa,1.125,0.156,0.551
Ensemble (alpha=0.87),0.820,0.552,0.721



Fraction of 1,000 bootstrap resamples where the ensemble beat the single best model (tuned XGBoost + ECFP):
  rmse: 1.000
  r2: 1.000
  spearman: 0.983

Verdict: ensembling helps on this held-out set -- reporting the real number either way, not adopting the ensemble just because it was tried.


### Update the comparison table and diagnostic plot

The ensemble is a real, bootstrap-robust improvement (not just a fluke of one fixed test set — 100% RMSE/R² win rate, 98.3% Spearman win rate over 1,000 resamples), so it's added to `results/model_comparison.csv` and `results/figures/model_diagnostics.png` as the best-performing option found in this project. **No model artifact overwrite here** — the ensemble is a weighted average of the two already-saved models (`results/models/xgb_ecfp.joblib`, `results/models/mlp_chemberta.joblib`) computed at inference time with the fixed weight `alpha` found above, not a new trained object requiring its own file. Reproducing it means running both models and combining their outputs with this alpha, a real (if modest) added serving cost worth stating plainly rather than glossing over.

In [27]:
ensemble_win_rates = bootstrap_compare(y_test, ensemble_preds, tuned_ecfp_preds, n_boot=1000, seed=0)

full_comparison = pd.DataFrame(
    {
        "XGBoost + ECFP (tuned)": tuned_metrics,
        "MLP + ChemBERTa": chemberta_metrics,
        f"Ensemble (alpha={best_alpha:.2f})": ensemble_metrics,
    }
).T
full_comparison.columns = ["RMSE", "R²", "Spearman ρ"]
full_comparison["bootstrap_win_rate_vs_best_single_model"] = [
    np.nan,  # XGBoost + ECFP (tuned) is itself the best single model being compared against
    np.nan,  # MLP + ChemBERTa already loses to XGBoost by construction (step 5); not re-compared here
    ensemble_win_rates["rmse"],
]

comparison_path = RESULTS_DIR / "model_comparison.csv"
full_comparison.to_csv(comparison_path)
print(f"Updated {comparison_path} with the ensemble:")
display(full_comparison.round(3))

fig = plot_diagnostics(
    y_test,
    {
        "XGBoost + ECFP (tuned)": tuned_ecfp_preds,
        "MLP + ChemBERTa": chemberta_preds,
        f"Ensemble (alpha={best_alpha:.2f})": ensemble_preds,
    },
    save_path=FIGURES_DIR / "model_diagnostics.png",
)
print(f"Re-saved {FIGURES_DIR / 'model_diagnostics.png'} with all three models.")

Updated ../results/model_comparison.csv with the ensemble:


,RMSE,R²,Spearman ρ,bootstrap_win_rate_vs_best_single_model
XGBoost + ECFP (tuned),0.830,0.541,0.714,NaN
MLP + ChemBERTa,1.125,0.156,0.551,NaN
Ensemble (alpha=0.87),0.820,0.552,0.721,1.0


Re-saved ../results/figures/model_diagnostics.png with all three models.


### Summary

**Step 1 — scaffold split:**
- 4,452 rows train / 1,113 rows test (80.0%/20.0% split), grouped by 814 train + 1,113 test Bemis-Murcko scaffolds rather than randomly.
- Zero scaffold overlap between train and test confirmed directly.
- Every compound's WT/D816V row pair confirmed to land entirely on one side of the split (never straddling), since they share identical scaffolds. In practice, all 925 compounds with both WT and D816V measurements happen to land in train under this split/seed — worth keeping in mind for Phase 5 (selectivity analysis), which relies on paired WT/D816V compounds.
- Split is seeded and deterministic (`seed=0`); a different seed changes which scaffold groups land in test while preserving the ~80/20 split size and the zero-leakage guarantee.

**Step 2 — XGBoost-on-ECFP baseline:**
- Trained on 4,452 compounds × 2,048-bit ECFP fingerprints, default hyperparameters (`src/models.py::train_xgboost_ecfp`).
- Sanity checks only: training R² = 0.809, held-out predictions non-degenerate (std = 0.841) and positively correlated with truth (Pearson r = 0.725).

**Step 3 — MLP-on-ChemBERTa comparison model:**
- Trained on 4,452 compounds × 768-dim frozen ChemBERTa embeddings (standardized first), default architecture, early-stopped at 42 iterations (`src/models.py::train_mlp_chemberta`).
- Sanity checks only: training R² = 0.748, held-out predictions non-degenerate (std = 1.174) and positively correlated with truth (Pearson r = 0.562).

**Step 4 — formal held-out evaluation (RMSE / R² / Spearman ρ), untuned:**

| Model | RMSE | R² | Spearman ρ |
| --- | --- | --- | --- |
| XGBoost + ECFP | 0.849 | 0.520 | 0.702 |
| MLP + ChemBERTa | 1.125 | 0.156 | 0.551 |

- Naive mean-baseline RMSE on the same held-out set: 1.233. Both models clearly beat it on RMSE and R² (verified directly, not just assumed).

**Step 5 — direct comparison:** bootstrapped the held-out set 1,000 times — XGBoost + ECFP beat MLP + ChemBERTa on RMSE, R², *and* Spearman ρ in **100% of resamples**. Likely why (dataset size favoring a sample-efficient tree ensemble, ECFP bits closer to direct pharmacophore signal than a frozen, non-fine-tuned embedding) discussed in the notebook, with an explicit scope caveat.

**Step 6 — saved artifacts:** models saved to `results/models/` (later overwritten by the tuned versions — see the addendum below); comparison table to `results/model_comparison.csv`.

**Step 7 — diagnostic plots:** predicted-vs-actual and residual plots saved to `results/figures/model_diagnostics.png` (later regenerated with the tuned model). Revealed XGBoost's predictions are compressed relative to the MLP's (tree-extrapolation limits) and that visible diagonal banding is a real data feature (Phase 2's censoring caps), not a plotting bug.

**Step 8 — addendum, variant-aware model (Phase 5 prerequisite):** the structure-only models can't distinguish a compound's WT vs. D816V rows (identical fingerprint either way). `add_variant_indicator()` appends a binary flag; retraining on the 2,049-feature input *improved* held-out metrics (RMSE 0.849→0.843) rather than hurting them, and the model now predicts genuinely different values per variant (imatinib WT=7.01 vs. D816V=6.80). Caveat: both anchor compounds are in the train set (no paired WT/D816V compound currently lands in test).

**Step 9 — addendum, hyperparameter tuning for the best legitimate accuracy:**
- `tune_xgboost_ecfp()` (added to `src/models.py`) runs `RandomizedSearchCV` (80 candidates × 5 folds) with `GroupKFold` grouped by Bemis-Murcko scaffold — so tuning doesn't repeat the leakage problem the outer scaffold split exists to prevent. Train set only; test set touched exactly once, at the end.
- **Final tuned results (held-out test set):**

| Model | RMSE | R² | Spearman ρ |
| --- | --- | --- | --- |
| XGBoost + ECFP (tuned) | **0.830** | **0.541** | **0.714** |
| XGBoost + ECFP, variant-aware (tuned) | **0.824** | **0.548** | **0.719** |
| MLP + ChemBERTa | 1.125 | 0.156 | 0.551 |

- A real, modest improvement (~2-3% RMSE) from tuning alone — consistent with the ~0.5–0.8 log-unit noise floor established in Phase 2 (ChEMBL's own cross-assay measurement disagreement): there's a hard ceiling on how far *any* model can legitimately go on this data, and this is close to it.
- Re-verified the Phase 5 anchor checks still hold with the tuned variant-aware model: imatinib's predicted WT-favoring shift is now 3.93× (up from 1.63× untuned — notably closer to the measured 9.07×), dasatinib's is 1.37× — still correctly the smaller of the two.
- Train RMSE (0.483) vs. test RMSE (0.830) — the gap is the honest cost of generalizing across a scaffold split, not overfitting hidden by a weaker evaluation.
- **The tuned models are now the production models** — they've replaced the untuned ones in `results/models/xgb_ecfp.joblib` and `xgb_ecfp_variant_aware.joblib`, and `results/model_comparison.csv` / `results/figures/model_diagnostics.png` were regenerated to match. Notebooks 05 and 06, which depend on the variant-aware model, need to be (and have been) re-run to pick up the improved model.

**Step 10 — addendum, ensembling XGBoost + ECFP with MLP + ChemBERTa:**
- `find_best_ensemble_weight()` (added to `src/models.py`) grid-searches the weight alpha for `alpha * XGBoost_pred + (1 - alpha) * MLP_pred`, selected via 5-fold **scaffold-grouped out-of-fold** predictions on the training set only (`sklearn.cross_val_predict` with the same `GroupKFold` used for tuning in step 9) — never against the test set, avoiding a subtler version of the same leakage the outer split and step 9's CV both guard against.
- Best weight found: **alpha = 0.87** (87% XGBoost, 13% MLP), OOF training RMSE 0.800 vs. 0.806 for XGBoost alone — a modest 0.8% OOF gain, consistent with the two models' errors being only partially uncorrelated (XGBoost + ECFP is simply the much stronger model here).
- **Final held-out test-set result — the best found in this project:**

| Model | RMSE | R² | Spearman ρ |
| --- | --- | --- | --- |
| XGBoost + ECFP (tuned) | 0.830 | 0.541 | 0.714 |
| MLP + ChemBERTa | 1.125 | 0.156 | 0.551 |
| **Ensemble (alpha=0.87)** | **0.820** | **0.552** | **0.721** |

- Real and bootstrap-robust, not a fluke of one fixed test set: the ensemble beat tuned XGBoost alone in 100% of 1,000 RMSE/R² bootstrap resamples and 98.3% of Spearman resamples.
- **Not adopted as a drop-in model-file replacement** — unlike step 9's tuning, this doesn't produce a single new artifact to overwrite `results/models/xgb_ecfp.joblib` with. It's a weighted average of the two already-saved models, computed at inference time; reproducing the ensemble means running both models and combining their outputs with alpha=0.87, a real (if modest) added serving cost stated explicitly rather than glossed over. `results/model_comparison.csv` and `results/figures/model_diagnostics.png` were updated to include it as the best-performing option found, alongside the two single models.
- No further legitimate gains were pursued past this point: remaining error is dominated by the ~0.5–0.8 log-unit noise floor established in Phase 2, not by anything left on the table for a scaffold-safe method to capture.

**All of Phase 4, plus all three addenda, are complete.**